In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from processor.text import TextProcessor

### Read and overview data

In [3]:
df = pd.read_excel('./input_data/review_full_text_vnmese_100.xlsx')

In [4]:
df.head(2)

,hotel_name,customer_name,title,full_content,rating,rating_New
0,Dal Vostro Hotel & Spa,Chrisinder,"Địa điểm, địa điểm, địa điểm ........",Vị trí tuyệt vời trong một con phố nhỏ cách Hồ...,bubble_50,5
1,Khách sạn Mường Thanh Grand Đà Nẵng,Lan thương,Đánh giá,Phục vụ tốt chăm sóc nhiệt tình. Buffer ngon n...,bubble_50,5


In [5]:
df.tail(2)

,hotel_name,customer_name,title,full_content,rating,rating_New
98,Hanoi Gravita Hotel,Jean K,Nơi tuyệt vời để ở lại để khám phá Hà Nội,"Vị trí tuyệt vời, khách sạn nằm trong khu phố ...",bubble_50,5
99,Khách sạn Tú Linh Palace,TorontoJose,"Decent tại chỗ, không quá ưa thích.",Tôi ở đây tổng cộng 5 đêm vào tháng Ba và thán...,bubble_40,4


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   hotel_name     100 non-null    object
 1   customer_name  100 non-null    object
 2   title          100 non-null    object
 3   full_content   100 non-null    object
 4   rating         100 non-null    object
 5   rating_New     100 non-null    int64 
dtypes: int64(1), object(5)
memory usage: 4.8+ KB


### Preprocessing

#### Lower case

In [7]:
df['clean_text'] = df.full_content.str.lower()

#### Check dup

In [8]:
df.duplicated(subset=['clean_text', 'rating_New']).any()

False

#### Check missing value

In [9]:
# If missing value, drop else next step

#### Replace multiple dot to 1 dot

In [10]:
df.clean_text = df.clean_text.str.replace('\.+', '.', regex=True)

#### Clean data

In [11]:
_processor = TextProcessor()

In [12]:
# clean text (replace emojicon, replace teencode, remove typo)
df.clean_text = df.clean_text.apply(lambda x: _processor.process_text(x))

In [13]:
# Remove punctuation and numbers
df.clean_text = df.clean_text.apply(lambda x: _processor.remove_punctuation_number(x))
df['clean_text'].head(5)

0    vị trí tuyệt vời trong một con phố nhỏ cách hồ...
1    phục vụ tốt chăm sóc nhiệt tình buffer ngon nh...
2    chúng tôi tìm kiếm một kỳ nghỉ ngân sách thấp ...
3    phòng và phòng tắm rất sạch sẽ giường ngủ thoả...
4    chúng tôi rất thích ở tại khách sạn riverside ...
Name: clean_text, dtype: object

In [14]:
# translate english to vietnamese
df.clean_text = df.clean_text.apply(lambda x: _processor.translate_english_to_vietnam(x))
df['clean_text'].head(5)

0    vị trí tuyệt vời trọng một con lừa phố nhỏ các...
1    phục vụ tốt chăm sóc nhiệt tình đệm ngon nhiều...
2    chúng tôi tìm kiếm một kỳ nghỉ ngân sách thấp ...
3    phòng và phòng tắm rất sạch sẽ giường ngủ thoả...
4    chúng tôi rất thích ở tại khách sạn ven sông s...
Name: clean_text, dtype: object

In [15]:
# Convert to unicode 
df.clean_text = df.clean_text.apply(lambda x: _processor.covert_unicode(x))
df['clean_text'].head(5)

0    vị trí tuyệt vời trọng một con lừa phố nhỏ các...
1    phục vụ tốt chăm sóc nhiệt tình đệm ngon nhiều...
2    chúng tôi tìm kiếm một kỳ nghỉ ngân sách thấp ...
3    phòng và phòng tắm rất sạch sẽ giường ngủ thoả...
4    chúng tôi rất thích ở tại khách sạn ven sông s...
Name: clean_text, dtype: object

In [16]:
# Filter pos-tagging
df.clean_text = df.clean_text.apply(lambda x: _processor.process_postag_thesea(x))
df['clean_text'].head(5)

0    tuyệt_vời trọng lừa nhỏ cách hoàn kiếm đi tuyệ...
1    tốt chăm_sóc đệm ngon nhiều đầy_đủ xem đẹp hướ...
2    tìm_kiếm nghỉ thấp chỉ trọng thành_lập chúng_t...
3    tắm rất sạch_sẽ ngủ thoải_mái gần khoảng lái_x...
4    rất thích sài_gòn bao_gồm truy_cập câu_lạc_bộ ...
Name: clean_text, dtype: object

In [17]:
# Remove stopword
df.clean_text = df.clean_text.apply(lambda x: _processor.remove_stopword(x))
df['clean_text'].head(5)

0    tuyệt_vời lừa hoàn kiếm tuyệt_vời tốt lừa giao...
1    tốt chăm_sóc đệm ngon đầy_đủ đẹp hướng hàn phụ...
2    tìm_kiếm nghỉ thành_lập hạnh_phúc tuyệt đẹp sạ...
3                 sạch_sẽ thoải_mái lái_xe dạo lây_lan
4    thích bao_gồm truy_cập câu_lạc_bộ nhẹ không_cồ...
Name: clean_text, dtype: object

In [20]:
# Remove na, remove empty string
df.clean_text.dropna(inplace=True)
df = df[df.clean_text != ''].reset_index(drop=True)
df.clean_text.head()

0    tuyệt_vời lừa hoàn kiếm tuyệt_vời tốt lừa giao...
1    tốt chăm_sóc đệm ngon đầy_đủ đẹp hướng hàn phụ...
2    tìm_kiếm nghỉ thành_lập hạnh_phúc tuyệt đẹp sạ...
3                 sạch_sẽ thoải_mái lái_xe dạo lây_lan
4    thích bao_gồm truy_cập câu_lạc_bộ nhẹ không_cồ...
Name: clean_text, dtype: object

### Feature Engineering

In [21]:
df.rating_New.value_counts()

5    69
4    18
3     5
1     5
2     3
Name: rating_New, dtype: int64

In [24]:
df['label'] = df.rating_New.apply(lambda x: 1 if x >=4 else 0)

In [25]:
df.label.value_counts()

1    87
0    13
Name: label, dtype: int64

#### Vectorize

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [27]:
vectorizer = TfidfVectorizer()

In [29]:
vectorized_data = vectorizer.fit_transform(df.clean_text)

In [31]:
df_vectorized = pd.DataFrame(vectorized_data.toarray(), columns=vectorizer.get_feature_names_out())
df_vectorized.head()

,ac,alova,ba_lô,bao_gồm,biệt_lập,bàn,bánh_mì,bánh_tét,báo,bãi_biển,...,ấm,ấm_cúng,ấn_tượng,ẩm,ẩm_ướt,ồn_ào,ổ_chuột,ổ_đĩa,ổn,ủi
0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.265677,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [32]:
vectorizer.get_feature_names_out()

array(['ac', 'alova', 'ba_lô', 'bao_gồm', 'biệt_lập', 'bàn', 'bánh_mì',
       'bánh_tét', 'báo', 'bãi_biển', 'bé', 'bình_minh', 'bít',
       'bóng_mát', 'bóng_râm', 'bùi_viện', 'băng', 'bơi', 'bạn_bè',
       'bảo_trì', 'bầu_không_khí', 'bẩn', 'bận_rộn', 'bận_tâm', 'bật',
       'bắt', 'bể_bơi', 'bỏ_qua', 'bổ_nhiệm', 'cao_điểm', 'chiên',
       'chu_đáo', 'chuyen', 'chuyên_nghiệp', 'chào', 'chào_mừng',
       'chào_đón', 'chân_thật', 'chén', 'chênh_lệch', 'chóng', 'chúc',
       'chăm_sóc', 'chăn', 'chạy', 'chảnh', 'chảy', 'chấp_nhận',
       'chất_lượng', 'chậm_chạp', 'chỉ_dẫn', 'chờ', 'chờ_đợi', 'chủ_nhân',
       'chủ_động', 'chứng_từ', 'co', 'cocktail', 'com', 'commisioned',
       'common', 'cà_phê', 'cài', 'cân_nhắc', 'câu_lạc_bộ', 'có_lẽ',
       'có_mặt', 'công_bố', 'công_tác', 'cũ', 'cũ_kỹ', 'cơ_bản', 'cư_ngụ',
       'cười', 'cải_thiện', 'cảm_tạ', 'cảm_xúc', 'cất_giữ', 'cậy',
       'cặn_kẽ', 'cỏ_khô', 'cố_gắng', 'cống_hiến', 'cổ', 'cởi_mở',
       'cửa_sổ', 'cực', 'dai', '

In [34]:
df_vectorized['label'] = df['label']

### Visualize

### Build model

#### Split into train/test

In [35]:
from sklearn.model_selection import train_test_split

In [36]:
X = df_vectorized.drop('label', axis=1)
y = df_vectorized[['label']]

In [37]:
X_train, X_test, y_train, y_test = train_test_split(X, y)

#### Train model

In [38]:
from sklearn.linear_model import LogisticRegression

In [39]:
model = LogisticRegression()

In [40]:
model.fit(X_train, y_train)

/Users/admin/.virtualenvs/liem_nguyen/lib/python3.9/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


LogisticRegression()

In [41]:
y_hat = model.predict(X_test)

### Evaluate result

In [42]:
accuracy = model.score(X_test, y_test)
print(f'Accuracy of model: {accuracy}')

Accuracy of model: 0.88


### Run new comments

In [43]:
df_comments = pd.DataFrame([
    {'comments': 'Khách sạn gần trung tâm, thuận lợi di chuyển, lễ tân phục vụ nhiệt tình.'},
    {'comments': 'Phòng cũ, dơ, cách âm kém. không hài lòng'}
])

In [46]:
df_comments_vectorized = pd.DataFrame(
    vectorizer.transform(df_comments.comments).toarray(), columns=vectorizer.get_feature_names_out())
df_comments_vectorized.head()

,ac,alova,ba_lô,bao_gồm,biệt_lập,bàn,bánh_mì,bánh_tét,báo,bãi_biển,...,ấm,ấm_cúng,ấn_tượng,ẩm,ẩm_ướt,ồn_ào,ổ_chuột,ổ_đĩa,ổn,ủi
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [47]:
model.predict(df_comments_vectorized)

array([1, 1])

### Language filter

In [48]:
!pip install langdetect


[notice] A new release of pip is available: 23.1.2 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [49]:
from langdetect import detect

In [50]:
df.clean_text.apply(lambda x: detect(x))

0     vi
1     vi
2     vi
3     vi
4     vi
      ..
95    vi
96    vi
97    vi
98    vi
99    vi
Name: clean_text, Length: 100, dtype: object

### Lemmatization

In [51]:
from nltk.stem import WordNetLemmatizer

In [52]:
lemma = WordNetLemmatizer()

In [53]:
lemma.lemmatize('eggs')

'egg'

In [54]:
lemma.lemmatize('worked', pos='v')

'work'